In [14]:
import pandas as pd
import io

import requests

In [55]:
def generate_airport_codes_df(output_filepath: str) -> None:
    """
    From a url of a csv file, download the file and clean it.
    Finally, export the cleaned data to a json file.
    """
    # get raw data
    url = "https://raw.githubusercontent.com/datasets/airport-codes/refs/heads/main/data/airport-codes.csv"
    response = requests.get(url)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.content.decode("utf-8")))

    # clean data
    df = df.dropna(subset=["iata_code"])
    df = df.loc[df["type"].str.endswith("_airport")]
    df["lat"] = df["coordinates"].str.split(", ", expand=True)[0].astype(float).round(5)
    df["lon"] = df["coordinates"].str.split(", ", expand=True)[1].astype(float).round(5)
    df = df.rename(columns={"ident": "icao_code"})
    df = df.drop(columns=["elevation_ft", "gps_code", "local_code", "coordinates"])
    df = df[["iata_code"] + [col for col in df.columns if col != "iata_code"]]
    df = df.sort_values("iata_code").reset_index(drop=True)
    df = df.set_index("iata_code")

    # export to json
    df.to_json(output_filepath, orient="index")


x = generate_airport_codes_df("src/flights/data/airports.json")
x

In [58]:
import json

DATA_FOLDER = "src/flights/data"

with open(f"{DATA_FOLDER}/airports.json", "r") as f:
    AIRPORTS_DATA = json.load(f)

In [59]:
AIRPORTS_DATA

{'AAA': {'icao_code': 'NTGA',
  'type': 'medium_airport',
  'name': 'Anaa Airport',
  'continent': 'OC',
  'iso_country': 'PF',
  'iso_region': 'PF-U-A',
  'municipality': 'Anaa',
  'lat': -17.3526,
  'lon': -145.51},
 'AAB': {'icao_code': 'YARY',
  'type': 'small_airport',
  'name': 'Arrabury Airport',
  'continent': 'OC',
  'iso_country': 'AU',
  'iso_region': 'AU-QLD',
  'municipality': 'Tanbar',
  'lat': -26.69639,
  'lon': 141.04872},
 'AAC': {'icao_code': 'HEAR',
  'type': 'medium_airport',
  'name': 'El Arish International Airport',
  'continent': 'AS',
  'iso_country': 'EG',
  'iso_region': 'EG-SIN',
  'municipality': 'El Arish',
  'lat': 31.07856,
  'lon': 33.83679},
 'AAD': {'icao_code': 'AAD',
  'type': 'small_airport',
  'name': 'Adado Airport',
  'continent': 'AF',
  'iso_country': 'SO',
  'iso_region': 'SO-GA',
  'municipality': 'Adado',
  'lat': 6.0958,
  'lon': 46.6375},
 'AAE': {'icao_code': 'DABB',
  'type': 'medium_airport',
  'name': 'Annaba Rabah Bitat Airport',
  

In [44]:
x.to_json("airport_codes.json", orient="records", lines=True)

In [40]:
x

'[{"ident":"03N","type":"small_airport","name":"Utirik Airport","continent":"OC","iso_country":"MH","iso_region":"MH-UTI","municipality":"Utirik Island","iata_code":"UTK","lat":11.22222,"lon":169.85143},{"ident":"07FA","type":"small_airport","name":"Ocean Reef Club Airport","continent":null,"iso_country":"US","iso_region":"US-FL","municipality":"Key Largo","iata_code":"OCA","lat":25.3254,"lon":-80.2748},{"ident":"07TE","type":"small_airport","name":"Cuddihy Field","continent":null,"iso_country":"US","iso_region":"US-TX","municipality":"Corpus Christi","iata_code":"CUX","lat":27.7211,"lon":-97.5128},{"ident":"0CO2","type":"small_airport","name":"Crested Butte Airpark","continent":null,"iso_country":"US","iso_region":"US-CO","municipality":"Crested Butte","iata_code":"CSE","lat":38.85192,"lon":-106.92834},{"ident":"0NM0","type":"small_airport","name":"Columbus Airport","continent":null,"iso_country":"US","iso_region":"US-NM","municipality":"Columbus","iata_code":"CUS","lat":31.8239,"lon"

In [32]:
x["coordinates"].str.split(", ")

0                   [11.222219, 169.851429]
1       [25.325399398804, -80.274803161621]
2                     [27.7211, -97.512802]
3                  [38.851918, -106.928341]
4                  [31.823898, -107.629924]
                       ...                 
8851                [38.965719, 121.538477]
8852         [42.2538888889, 125.703333333]
8853                [41.639801, 123.483002]
8854         [42.8828010559, 129.451004028]
8855                  [40.542524, 122.3586]
Name: coordinates, Length: 8856, dtype: object

In [18]:
x["type"].unique()

array(['heliport', 'small_airport', 'closed', 'seaplane_base',
       'balloonport', 'medium_airport', 'large_airport'], dtype=object)